In [0]:
from pyspark.sql.functions import current_timestamp

# Get all validated tables for this run
validation_df = spark.sql(f"""

SELECT table_name
FROM workspace.audit.validation_summary

""")

tables = [row["table_name"] for row in validation_df.collect()]

# Update silver counts
for bronze_table in tables:
    # demo_bronze.customers -> demo_silver.customers

    silver_table = bronze_table.replace(
        "demo_bronze",
        "demo_silver"
    )

    print(f"Processing {silver_table}")

    # Get silver count
    silver_count = (
        spark.sql(f"""
            SELECT COUNT(*) as cnt
            FROM silver.{silver_table}
        """)
        .collect()[0]["cnt"]
    )

    print(f"Silver Count = {silver_count}")

    # Update validation table
    spark.sql(f"""

    UPDATE workspace.audit.validation_summary
    SET
        silver_count = {silver_count}
    WHERE table_name = '{bronze_table}'

    """)

print("Silver counts updated successfully")

In [0]:
%sql
select * from workspace.audit.validation_summary

In [0]:
%sql
Select 'Duplicates in brnz_customer -->' as table_name,count(*) from (
Select customer_id,count(*) from workspace.demo_bronze.customers group by customer_id having count(*) > 1
)
union all
Select 'Duplicates in silver_customer-->' as table_name,count(*) from (
Select customer_id,count(*) from workspace.demo_silver.customers group by customer_id having count(*) > 1
)